# Explicit representations of measurement irregularity in deep learning prediction of sepsis mortality

Analysis notebook for *Explicit Representations of Measurement Irregularity in Deep Learning Prediction of Sepsis Mortality: A Controlled Evaluation on MIMIC-IV* (International Journal of Intelligent Engineering and Systems, Paper No. 20267589). Throughout, **[15]** is the earlier study whose cohort, partition, architecture, and hyperparameters are reused; the README gives the reference.

**What the notebook does**

* Rebuilds the hourly values and observation masks together from time-stamped records, with one set of rules for hour assignment, validity, and aggregation, and without backward filling.
* Retrains every model on the partitions of [15], the reference R2 included. Early stopping monitors an internal 15% of the training patients, never the patients being evaluated.
* Compares five input representations (R1 to R5), a timing-only model (T), cumulative measurement counts (C), static inputs only (S), the GRU-D family, and a time-aware attention network (TBAL).
* Repeats the main contrast in a later admission period under an analysis plan fixed before training.
* Reports discrimination, calibration, paired comparisons, bounded differences, operating points, and net benefit.

| Cell | Content | Runtime |
|---|---|---|
| 1–3 | Setup, modules, and the inputs of [15] | CPU |
| 4 | Index ICU stay and eligibility at the 24-hour landmark | CPU |
| 5–6 | Rebuilt values and masks, the training table (written once), reconciliation | CPU |
| 7 | Derived channels and the cutoff test | CPU |
| 8 | Partitions and the internal early-stopping splits | CPU |
| 9 | Audits: winsorisation, static inputs, timing of the predictors | CPU |
| 10 | Optional: reproduction of the training of [15] | GPU |
| 11 | Model definitions | CPU |
| 12 | R1 to R5, T, C, and S | GPU |
| 13 | Random search for GRU-D and TBAL | GPU |
| 14 | GRU-D family and TBAL | GPU |
| 15 | Earlier inputs, and models without the Charlson index | GPU |
| 16–17 | Temporal validation | GPU |
| 18–21 | Performance, comparisons, operating points, measurement counts | CPU |
| 22 | Figures | CPU |
| 23 | Export of aggregate results | CPU |

The training cells also run on a CPU runtime, more slowly; the run log records the device of every run.

**How to run**

1. Prepare the files described in the README and set `BASE` in Cell 1.
2. Run the cells in order. Every finished fold and run is saved at once, so rerunning a cell after an interruption resumes where it stopped. Each model is trained in its own short-lived Python process, so memory does not build up over a long session (Cell 8).
3. In a new session, run Cells 1 to 8 and Cell 11, then the cell to be continued. Cell 5 then reads the training table already written, so the raw extracts are not needed again; Cells 9 and 10 can be skipped.

**Patient-level data.** The training table (`paperA_rev_hourly.csv`), the caches, and the predictions hold patient-level data. They stay in `BASE` and are never exported, as the MIMIC-IV data use agreement requires. Cell 23 exports aggregate tables, figures, code, and run logs only.

**Checks that stop the notebook**

* agreement of the static inputs with the arrays of [15] (Cell 3);
* the cutoff test (Cell 7);
* reproduction of the training of [15] (Cell 10, optional).

## 1. Setup

Set `BASE` to the folder that holds the files described in the README. In Colab this is a folder on Google Drive; elsewhere the mount is skipped.

In [ ]:
# ============================================================
# 1. Session setup
# ============================================================
import os, sys, json, time, glob, warnings
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')     # quieter TensorFlow
import numpy as np
import pandas as pd

try:                                   # Colab; elsewhere set BASE to the data folder
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

BASE = '/content/drive/MyDrive/DL_sepsis_mortality'     # the folder that holds the inputs
SRC  = f'{BASE}/src'

# Outputs of [15] and the optional mask of the first submission: read, never modified.
CACHE_B = f'{BASE}/cache_rebuilt'
RES_B   = f'{BASE}/results_rebuilt'
MASK_OLD_FILE = f'{BASE}/cache_paperA_v2/mask.npz'

# Revision outputs. CACHE_R and PRED_R hold patient-level files and are
# never exported.
CACHE_R = f'{BASE}/cache_paperA_rev'
RES_R   = f'{BASE}/results_paperA_rev'
PRED_R  = f'{RES_R}/preds'
TAB_R   = f'{BASE}/tables_paperA_rev'
FIG_R   = f'{BASE}/figures_paperA_rev'
for d in (CACHE_R, RES_R, PRED_R, TAB_R, FIG_R):
    os.makedirs(d, exist_ok=True)
sys.path.insert(0, SRC)

SEEDS = [42, 1, 2]
EPOCHS, BATCH, PATIENCE = 40, 512, 4
STOP_FRAC = 0.15     # internal split for early stopping (and Platt scaling)
N_TRIALS = 15        # random-search budget of [15]
MARGIN = 0.01        # AUROC tolerance, fixed before the temporal analysis
LOG = f'{RES_R}/runs.csv'
print('ready')


## 2. Modules and configuration

The notebook loads `paper_a_rev.py`, `windows.py`, `train.py`, and `metrics.py` from `BASE/src`, and the analysis plan from `BASE/config`. When they are missing, this cell copies them from the repository.

In [ ]:
# ============================================================
# 2. Modules
# ============================================================
import shutil
import subprocess

REPO = 'https://github.com/IrawanIren/sepsis-measurement-irregularity'
MODULES = ('paper_a_rev.py', 'windows.py', 'train.py', 'metrics.py')
need_src = [m for m in MODULES if not os.path.exists(f'{SRC}/{m}')]
need_cfg = not os.path.exists(f'{BASE}/config/analysis_plan_temporal.json')
if need_src or need_cfg:
    clone = '/tmp/sepsis-measurement-irregularity'
    if not os.path.exists(clone):
        subprocess.run(['git', 'clone', '--depth', '1', REPO, clone], check=True)
    os.makedirs(SRC, exist_ok=True)
    for m in need_src:
        shutil.copy(f'{clone}/src/{m}', f'{SRC}/{m}')
    if need_cfg:
        shutil.copytree(f'{clone}/config', f'{BASE}/config', dirs_exist_ok=True)
print('modules in', SRC, ':', ', '.join(MODULES))
print('configuration in', f'{BASE}/config')


## 3. Modules, the inputs of [15], and the ICU stays

The static inputs are rebuilt from the cohort table with exactly the encoding of `windows.build_arrays` and checked against the arrays of [15]; the cell stops if they differ.

In [ ]:
# ============================================================
# 3. Modules, the inputs of [15], and the ICU stays
# ============================================================
import importlib.util
from sklearn.preprocessing import OneHotEncoder


def load(name):
    """Load a module straight from its file; a mounted Drive can serve a
    stale copy through the normal import cache."""
    spec = importlib.util.spec_from_file_location(name, f'{SRC}/{name}.py')
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module


mt = load('metrics')
wn = load('windows')
tr = load('train')
pa = load('paper_a_rev')

import tensorflow as tf
import keras
tf.get_logger().setLevel('ERROR')     # hides retracing notices from repeated model builds
GPUS = tf.config.list_physical_devices('GPU')
for g in GPUS:                        # the notebook itself trains only in Cell 10, so it
    try:                              # takes GPU memory only as needed and leaves the rest
        tf.config.experimental.set_memory_growth(g, True)   # to the training processes
    except Exception:
        pass
GPU = bool(GPUS)
# Same precision policy as [15] on GPU; float32 on CPU, where float16 is slow.
keras.mixed_precision.set_global_policy('mixed_float16' if GPU else 'float32')
DEVICE = 'GPU' if GPU else 'CPU'
print('GPU available:', GPU, '| TensorFlow', tf.__version__, '| Keras', keras.__version__)

cfg = json.load(open(f'{BASE}/feature_config_rebuilt.json'))
ID, LABEL, HOUR = cfg['id_col'], cfg['label_col'], cfg['hour_col']
DYN = cfg['dynamic_features']
N_DYN = len(DYN)
NUM, CAT = cfg['static_numeric'], cfg['static_categorical']
MASKED = [f for f in DYN if f not in pa.UNMASKED]
URINE = [f for f in DYN if f in pa.UNMASKED][0]
idx_m = [DYN.index(f) for f in MASKED]
print(f'{N_DYN} dynamic variables ({len(MASKED)} with a mask); static: '
      f'{len(NUM)} numeric, {len(CAT)} categorical')
print('static numeric:', NUM)
print('static categorical:', CAT)

df = (pd.read_parquet(f'{BASE}/sepsis_rebuilt_24h.parquet')
        .sort_values([ID, HOUR]).reset_index(drop=True))
folds = pd.read_csv(f'{BASE}/cv_folds.csv')
test_ids = pd.read_csv(f'{BASE}/test_set_ids.csv')[ID].values
BEST_HP = json.load(open(f'{BASE}/best_hp_rebuilt.json'))
BW = json.load(open(f'{BASE}/best_window_rebuilt.json'))
W, S = BW['window_size'], BW['step']

z = np.load(f'{CACHE_B}/arrays_h24.npz', allow_pickle=True)
dyn_b, stat_b, pids = z['dyn'], z['stat'], z['pids']
y_all = z['y'].astype(np.float32)
row = {p: i for i, p in enumerate(pids)}
N = len(pids)
assert dyn_b.shape == (N, 24, 2 * N_DYN), dyn_b.shape

# Static inputs of [15] by name: numeric columns, then one-hot categories,
# encoded exactly as in windows.build_arrays.
first = df.groupby(ID, sort=True).first().loc[pids]
stat_num0 = first[NUM].to_numpy(dtype=float)
enc = OneHotEncoder(sparse_output=False, dtype=np.float32, handle_unknown='ignore')
stat_cat = enc.fit_transform(first[CAT])
rebuilt_b = np.hstack([stat_num0, stat_cat])
gap = float(np.nanmax(np.abs(rebuilt_b - stat_b)))
assert np.allclose(rebuilt_b, stat_b, atol=1e-4, equal_nan=True), \
    f'static inputs do not match the arrays of [15] (largest gap {gap:.4g})'
CAT_LEVELS = [f'{c}={v}' for c, vals in zip(CAT, enc.categories_) for v in vals]

# ICU stays; which stay produced each patient's row is settled in Cell 4.
p_icu = f'{BASE}/icustay_detail.csv'
icu = pd.read_csv(p_icu, sep=pa.detect_sep(p_icu))
icu = icu[icu.subject_id.isin(pids)].copy()

mask_old = None
if os.path.exists(MASK_OLD_FILE):
    zm = np.load(MASK_OLD_FILE, allow_pickle=True)
    if list(zm['features']) == MASKED and np.array_equal(zm['pids'], pids):
        mask_old = zm['mask'].astype(np.float32)
print(f'cohort {N:,} | window W={W}, S={S} | hyperparameters {BEST_HP}')
print('first-submission mask:', 'loaded' if mask_old is not None
      else 'not found (R3_old and the mask comparison are skipped)')


## 4. Index ICU stay and eligibility at the 24-hour landmark

**Index stay.** The cohort query of [15] keeps every Sepsis-3 stay that is the first ICU stay of its hospital admission and lasts at least 24 hours. Because `first_icu_stay` is defined per admission, a patient can have more than one such stay, and the first ICU stay of the patient can belong to another admission. For each patient, the cell takes the stay that meets the criteria of [15]:

* the `stay_id` column of the cohort table, when the table has one;
* otherwise, among several candidates, the one whose SOFA score at onset, age, and outcome agree with the table;
* when these do not decide, the earliest.

The cell also reports how many patients have a different stay from the mapping of the first submission (the first ICU stay of the patient).

**Landmark:** 24 hours after the clock hour of ICU admission, the end of the last hour that any input can use.

**Eligibility** describes the cohort; it excludes no one, and all patients of [15] are analysed. A patient is counted as eligible when the antibiotic, the culture, and SOFA ≥ 2 had all occurred by the landmark, so that Sepsis-3 was established at the time of prediction, and the patient was alive and still in intensive care at that time. A patient with any of these times missing is counted as not eligible. Two looser criteria are reported for comparison.

The cell reads `sepsis3_timing.csv`, the result of `sql/sepsis3_timing.sql`. It also sets the calendar period of each patient from `anchor_year_group`, which defines the temporal split.

In [ ]:
# ============================================================
# 4. Landmark eligibility
# ============================================================
PATH_S3 = f'{BASE}/sepsis3_timing.csv'
timing = pd.read_csv(PATH_S3, sep=pa.detect_sep(PATH_S3))

# (a) The ICU stay behind each patient's row in the cohort table of [15].
stays, stay_report = pa.resolve_stays(pids, first, timing, icu, LABEL)
stay_report.to_csv(f'{TAB_R}/stay_resolution.csv', index=False)
print(stay_report.to_string(index=False))
first_any = (icu[pa.truthy(icu['first_icu_stay']).to_numpy()].sort_values('icu_intime')
               .drop_duplicates('subject_id').set_index('subject_id')['stay_id'].reindex(pids))
STAY_CHANGED = first_any.to_numpy() != stays
n_diff = int(STAY_CHANGED.sum())
print(f'patients whose stay differs from the first ICU stay of the patient '
      f'(the mapping of the first submission): {n_diff:,} ({100 * n_diff / N:.1f}%)')
icu_s = icu.drop_duplicates('stay_id').set_index('stay_id')
stay_row = pd.Series(np.arange(N), index=stays)
intime_of = pd.to_datetime(icu_s['icu_intime']).reindex(stays)
hadm_to_stay = pd.Series(stays, index=icu_s['hadm_id'].reindex(stays).to_numpy())
los_h = (pd.to_datetime(icu_s['icu_outtime']).reindex(stays) - intime_of).dt.total_seconds() / 3600
print(f'ICU stays shorter than 24 h among the chosen stays: {int((los_h < 24).sum())}')

# (b) Sepsis-3 times relative to the landmark.
lt = pa.landmark_table(timing, stays).set_index('stay_id').reindex(stays)
n_missing = int(lt['icu_intime'].isna().sum())
print(f'cohort stays without Sepsis-3 times: {n_missing:,}')

# Eligibility describes the cohort; every patient is analysed.
LANDMARK_ELIGIBLE = lt['eligible'].eq(True).to_numpy()
# Calendar period of each patient (anchor_year_group is recorded per patient).
year = pd.to_numeric(timing.drop_duplicates('subject_id').set_index('subject_id')['anchor_year_group']
                     .astype(str).str.extract(r'(\d{4})')[0], errors='coerce')
PERIOD = year.reindex(pids).to_numpy(dtype=float)
print(f'calendar period unknown for {int(np.isnan(PERIOD).sum()):,} patients')
died = y_all == 1


def line(label, mask):
    return {'group': label, 'patients': int(mask.sum()), 'share (%)': round(100 * mask.mean(), 2),
            '30-day mortality (%)': round(100 * died[mask].mean(), 2) if mask.any() else np.nan}


summary = pd.DataFrame([
    line('whole cohort of [15]', np.ones(N, bool)),
    line('eligible: antibiotic, culture and SOFA >= 2 by the landmark', LANDMARK_ELIGIBLE),
    line('suspicion of infection and SOFA >= 2 by the landmark',
         lt['eligible_suspicion_and_sofa'].eq(True).to_numpy()),
    line('SOFA >= 2 by the landmark', lt['eligible_sofa_only'].eq(True).to_numpy()),
    line('not eligible', ~LANDMARK_ELIGIBLE),
    line('died by the landmark', lt['died_by_landmark'].eq(True).to_numpy()),
    line('left the ICU before the landmark', lt['left_icu_by_landmark'].eq(True).to_numpy()),
])
summary.to_csv(f'{TAB_R}/landmark_eligibility.csv', index=False)
print(summary.to_string(index=False))

hours = lt[['antibiotic_h', 'culture_h', 'suspicion_h', 'sofa_h', 'complete_h']]
q = hours.describe(percentiles=[.05, .25, .5, .75, .95]).T.round(1)
q.to_csv(f'{TAB_R}/sepsis3_timing_vs_landmark.csv')
print('\nhours relative to the landmark (negative: before it)')
print(q.to_string())

late = pd.DataFrame({
    'antibiotic after the landmark': hours['antibiotic_h'] > 0,
    'culture after the landmark': hours['culture_h'] > 0,
    'SOFA >= 2 after the landmark': hours['sofa_h'] > 0,
    'a time missing': hours[['antibiotic_h', 'culture_h', 'sofa_h']].isna().any(axis=1),
    'died by the landmark': lt['died_by_landmark'].eq(True),
    'left the ICU before the landmark': lt['left_icu_by_landmark'].eq(True)})[~LANDMARK_ELIGIBLE]
late.sum().rename('patients not eligible').to_csv(f'{TAB_R}/landmark_exclusion_reasons.csv')
print('\nreasons among the patients not eligible (a patient can have more than one)')
print(late.sum().to_string())

period = pd.crosstab(pd.Series(PERIOD, name='anchor_year_group start'),
                     pd.Series(np.where(LANDMARK_ELIGIBLE, 'eligible', 'not eligible'), name=''))
period.to_csv(f'{TAB_R}/period_by_eligibility.csv')
print('\n', period.to_string())
pd.DataFrame({ID: pids, 'stay_id': stays, 'eligible': LANDMARK_ELIGIBLE, 'period': PERIOD}).to_csv(
    f'{CACHE_R}/cohort_rev.csv', index=False)          # patient level: stays on Drive


## 5. Rebuilt values and masks, and the training table

One set of rules applies to the values and the masks alike:

* **Sources.** Vital signs and bedside glucose come from the vital-sign extract, laboratory results from the laboratory extract, and lactate from the lactate extract, which is recorded per hospital admission and assigned to the index ICU stay. Glucose pools the laboratory and bedside records, so its value and its mask follow the same rule.
* **Hours.** Each record is assigned to a clock hour counted from the clock hour of ICU admission, and only hours 0 to 23 are kept, so no record at or after the landmark can enter.
* **Validity.** A zero is treated as a missing result for the laboratory variables and lactate. Other values are clipped to the bounds of [15], taken from the feature configuration when it lists them and otherwise from the range of the cohort table of [15].
* **Aggregation.** The value of an hour is the mean of its valid records, and the mask is one when the hour holds at least one valid record.
* **No backward filling.** Hours before the first observation stay missing; they are filled per fold with the median of the training patients.

Urine output has no mask, and an hour without a record holds zero. When the urine-output extract is present, the hourly values are rebuilt from it (the sum or the mean of the records, whichever agrees with [15]); otherwise the values of [15] are used.

**Training table: `paperA_rev_hourly.csv`** in `BASE`, in the format of the cohort table of [15]: one row per patient and hour (19,902 × 24 rows).

| Column | Content |
|---|---|
| `subject_id`, `stay_id`, `hour`, `label` | identifiers, hour, outcome |
| `eligible`, `period`, `split` | landmark eligibility, calendar period, partition of [15] |
| `<variable>` | carried-forward value, empty before the first observation |
| `m_<variable>` | observation mask |
| `urineoutput` | urine output |
| 16 numeric and 2 categorical columns | static inputs |

* The table is written once (about 130 MB). Later sessions read it directly, so the raw extracts are no longer needed.
* Every model is trained on the content of this file, in the first session too: once written, the table is read back.
* The derived channels (Δx, δ, d, cumulative counts) are computed from the value and mask columns in Cell 7.
* The meaning of every column is written to `tables_paperA_rev/training_table_dictionary.csv`.

**This file holds patient-level data.** Keep it in `BASE` and do not share it; Cell 23 does not export it.

In [ ]:
# ============================================================
# 5. Raw extracts and the new training table: set the paths
# ============================================================
PATH_VITAL   = f'{BASE}/vitalsign.csv'        # vital signs and bedside glucose
PATH_LAB     = f'{BASE}/pivot_lab.csv'
PATH_LACTATE = f'{BASE}/lactate.csv'
PATH_URINE   = f'{BASE}/urine_output.csv'     # optional

TRAIN_CSV  = f'{BASE}/paperA_rev_hourly.csv'        # the new training table (patient level)
TRAIN_META = f'{BASE}/paperA_rev_hourly_meta.json'

missing = [p for p in (PATH_VITAL, PATH_LAB, PATH_LACTATE) if not os.path.exists(p)]
if os.path.exists(TRAIN_CSV) and os.path.exists(TRAIN_META):
    print(f'training table found ({os.path.getsize(TRAIN_CSV) / 1e6:.0f} MB); '
          'the raw extracts are not needed')
elif missing:
    print('Not found:', *missing, sep='\n  ')
    print('\nCSV files in BASE:')
    for f in sorted(glob.glob(f'{BASE}/*.csv')):
        print(f'  {os.path.basename(f):<42}{os.path.getsize(f) / 1e6:9.1f} MB')
    raise FileNotFoundError('set the paths above')
else:
    print('extracts found | urine output extract:', 'yes' if os.path.exists(PATH_URINE) else 'no')


In [ ]:
# ============================================================
# 5b. Build the aligned grid once and write the training table
# ============================================================
STATIC_LABS = [f for f in pa.STATIC_LABS if f in NUM]

# Partition of each patient in [15]: fold k or the held-out test set.
fold_num = (pd.Series(folds['fold'].to_numpy(), index=folds[ID].to_numpy().astype(pids.dtype))
              .reindex(pids).to_numpy(dtype=float))
SPLIT = np.full(N, 'excluded', dtype=object)
SPLIT[np.isin(pids, test_ids)] = 'test'
in_dev = ~np.isin(pids, test_ids) & ~np.isnan(fold_num)
SPLIT[in_dev] = [f'fold {int(k)}' for k in fold_num[in_dev]]


def validity_bounds():
    """Bounds of [15]: from the feature configuration when it lists them,
    otherwise the range of each variable in the cohort table, which
    already reflects the clipping of [15]."""
    for key in ('clip_bounds', 'bounds', 'clipping', 'physiological_bounds'):
        if isinstance(cfg.get(key), dict):
            return {k: tuple(v) for k, v in cfg[key].items()}, f'feature configuration ({key})'
    b = {f: (float(np.nanmin(dyn_b[:, :, DYN.index(f)])), float(np.nanmax(dyn_b[:, :, DYN.index(f)])))
         for f in DYN}
    for f in STATIC_LABS:
        v = stat_num0[:, NUM.index(f)]
        b[f] = (float(np.nanmin(v)), float(np.nanmax(v)))
    return b, 'range of the cohort table of [15]'


if not (os.path.exists(TRAIN_CSV) and os.path.exists(TRAIN_META)):
    BOUNDS, src_b = validity_bounds()
    pd.DataFrame([{'variable': k, 'lower': v[0], 'upper': v[1]} for k, v in BOUNDS.items()]
                 ).to_csv(f'{TAB_R}/validity_bounds.csv', index=False)
    print('validity bounds from the', src_b)

    vital_cols = [f for f in MASKED if f in pa.VITAL] + (['glucose'] if 'glucose' in MASKED else [])
    lab_cols = [f for f in MASKED if f in pa.LAB] + STATIC_LABS
    t0 = time.time()
    vs, vs_cols, vs_absent = pa.read_extract(PATH_VITAL, 'stay_id', vital_cols, stays)
    lab, lab_cols_p, lab_absent = pa.read_extract(PATH_LAB, 'stay_id', lab_cols, stays)
    lac, _, _ = pa.read_extract(PATH_LACTATE, 'hadm_id', ['lactate'], hadm_to_stay.index)
    lac['stay_id'] = lac['hadm_id'].map(hadm_to_stay)
    lac = lac.dropna(subset=['stay_id']).astype({'stay_id': 'int64'})
    print(f'read in {time.time() - t0:.0f}s: vital {len(vs):,} rows, laboratory {len(lab):,}, '
          f'lactate {len(lac):,}')
    if vs_absent or lab_absent:
        print('columns absent: vital', vs_absent, '| laboratory', lab_absent)

    parts, audit = [], []
    for name, d, cols in (('vital signs and bedside glucose', vs, vs_cols),
                          ('laboratory', lab, [c for c in lab_cols_p if c in MASKED]),
                          ('lactate', lac, ['lactate'])):
        sums, counts, info = pa.accumulate(d, 'stay_id', cols, stay_row, intime_of, N, BOUNDS)
        parts.append((sums, counts))
        audit.append({'source': name, **info})
    V, M, _ = pa.combine(parts, MASKED, N)

    found = pa.first_values(lab, 'stay_id', [c for c in STATIC_LABS if c in lab_cols_p],
                            stay_row, intime_of, N, BOUNDS)
    stat_new = stat_num0.copy()
    for f, (vals, _) in found.items():
        stat_new[:, NUM.index(f)] = vals      # first result in hours 0-23; missing stays missing
    if len(found) < len(STATIC_LABS):
        print('static laboratory inputs kept from [15]:', sorted(set(STATIC_LABS) - set(found)))

    U_old = dyn_b[:, :, DYN.index(URINE)]
    URINE_SOURCE = 'cohort table of [15]'
    U_new = U_old.astype(np.float32)
    if os.path.exists(PATH_URINE):
        ur, ur_cols, _ = pa.read_extract(PATH_URINE, 'stay_id', [URINE], stays)
        if ur_cols:
            total, mean, info = pa.hourly_total(ur, 'stay_id', URINE, stay_row, intime_of, N, BOUNDS)
            audit.append({'source': 'urine output', **info})
            agree = {'sum': np.mean(np.isclose(total, U_old, rtol=1e-3, atol=1e-3)),
                     'mean': np.mean(np.isclose(mean, U_old, rtol=1e-3, atol=1e-3))}
            how = max(agree, key=agree.get)
            U_new = total if how == 'sum' else mean
            URINE_SOURCE = f'rebuilt (hourly {how}; agreement with [15] {100 * agree[how]:.1f}%)'
    print('urine output:', URINE_SOURCE)

    audit = pd.DataFrame(audit)
    audit.to_csv(f'{TAB_R}/source_audit.csv', index=False)
    assert (audit['records at or after the landmark kept'] == 0).all()

    table = pa.training_table(pids, stays, y_all, V, M, U_new, stat_new, first[CAT].to_numpy(),
                              NUM, CAT, MASKED, URINE, LANDMARK_ELIGIBLE, PERIOD, SPLIT, id_col=ID)
    table.to_csv(TRAIN_CSV, index=False, float_format='%.6g')
    json.dump({'written': time.strftime('%Y-%m-%d %H:%M:%S'), 'patients': int(N),
               'rows': int(len(table)), 'columns': list(table.columns),
               'validity bounds': src_b, 'urine_source': URINE_SOURCE},
              open(TRAIN_META, 'w'), indent=2)
    del table, vs, lab, lac, parts
    print(f'training table written in {time.time() - t0:.0f}s: {TRAIN_CSV} '
          f'({os.path.getsize(TRAIN_CSV) / 1e6:.0f} MB)')

# Every session, the first one included, trains on the table as written.
tt = pa.read_training_table(TRAIN_CSV, ID, pids, MASKED, URINE, NUM)
V, M, U_new = tt['V'], tt['M'], tt['U']
static_first = {f: tt['stat_num'][:, NUM.index(f)] for f in STATIC_LABS}
URINE_SOURCE = json.load(open(TRAIN_META))['urine_source']
assert np.array_equal(tt['stay_id'].astype('int64'), stays), \
    'the training table was built from other ICU stays; delete paperA_rev_hourly.csv and its meta file, then rerun'
if not (np.array_equal(tt['eligible'], LANDMARK_ELIGIBLE) and (tt['split'] == SPLIT.astype(str)).all()):
    # Eligibility and partition changed since the table was written (Cell 4):
    # update those columns only; values and masks stay as they are.
    t_ = pd.read_csv(TRAIN_CSV, low_memory=False).sort_values([ID, 'hour'], kind='mergesort')
    t_['eligible'] = np.repeat(LANDMARK_ELIGIBLE.astype(int), 24)
    t_['period'] = np.repeat(PERIOD, 24)
    t_['split'] = np.repeat(SPLIT, 24)
    t_.to_csv(TRAIN_CSV, index=False, float_format='%.6g')
    del t_
    print('eligibility and partition updated in the training table (values and masks unchanged)')
pa.table_dictionary(MASKED, URINE, NUM, CAT, STATIC_LABS, URINE_SOURCE, id_col=ID).to_csv(
    f'{TAB_R}/training_table_dictionary.csv', index=False)
del tt
print(f'training table loaded: {N:,} patients x 24 hours | urine output: {URINE_SOURCE}')
print('hours with a measurement: '
      + ', '.join(f'{f} {100 * M[:, :, j].mean():.0f}%' for j, f in enumerate(MASKED)))


## 6. Reconciliation with the cohort table of [15]

For every variable, over all hours and over the final window (hours 18 to 23), the rebuilt inputs are compared with the cohort table of [15]:

* **Measured hours** (rebuilt mask one): whether the table of [15] holds the same value, and how many are true repeats, that is, a new measurement equal to the previous one.
* **Unmeasured hours after the first observation:** whether the table of [15] carries the previous value forward or holds a value without a record in the rebuilt extraction.
* **Hours before the first observation:** filled backward in the table of [15], left missing in the rebuilt inputs.

The last column must be 100%: in the rebuilt inputs, every unmeasured hour equals the hour before it.

In [ ]:
# ============================================================
# 6. Reconciliation with the cohort table of [15]
# ============================================================
X_old = dyn_b[:, :, idx_m]
rec_all = pa.reconcile(V, M, X_old, MASKED, M_old=mask_old)
rec_win = pa.reconcile(V, M, X_old, MASKED, hours=range(24 - W, 24), M_old=mask_old)
rec_all.to_csv(f'{TAB_R}/reconciliation_by_variable.csv', index=False)
rec_win.to_csv(f'{TAB_R}/reconciliation_final_window.csv', index=False)
pd.set_option('display.width', 250)
show = ['variable', 'measured hours (%)', 'measured: table value equal (%)',
        'measured after an earlier one: true repeat (%)',
        'unmeasured after first: table carried forward (%)',
        'table change without source record (% of hours)',
        'before first measurement (% of hours)', 'rebuilt: unmeasured hours equal to previous (%)']
print('[all hours]')
print(rec_all[show].round(1).to_string(index=False))
print(f'\n[final window, hours {24 - W}-23]')
print(rec_win[show].round(1).to_string(index=False))
assert (rec_all['rebuilt: unmeasured hours equal to previous (%)'].dropna() > 99.999).all()
if mask_old is not None:
    # The first submission linked each patient to the first ICU stay of the
    # patient. Where that was another stay, its mask cannot match the values.
    am = (mask_old == M).mean(axis=1)                      # patients x variables
    by = pd.DataFrame({'variable': MASKED,
                       'stay unchanged: hours equal (%)': 100 * am[~STAY_CHANGED].mean(axis=0),
                       'stay changed: hours equal (%)': 100 * am[STAY_CHANGED].mean(axis=0)
                       if STAY_CHANGED.any() else np.nan})
    by.to_csv(f'{TAB_R}/mask_first_submission_vs_rebuilt.csv', index=False)
    print(f'\nfirst-submission mask against the rebuilt mask, by whether the stay changed '
          f'({int((~STAY_CHANGED).sum()):,} unchanged, {int(STAY_CHANGED.sum()):,} changed):')
    print(by.round(1).to_string(index=False))
# Per patient: do the measured hours of the rebuilt grid carry the values
# of the cohort table? A wrongly chosen ICU stay shows up as a patient
# whose measured hours mostly disagree.
VIT = [j for j, f in enumerate(MASKED) if f in pa.VITAL]
obs = M[:, :, VIT] == 1
eq = np.isclose(X_old[:, :, VIT], V[:, :, VIT], rtol=1e-3, atol=1e-3) & obs
agree = eq.sum(axis=(1, 2)) / np.maximum(obs.sum(axis=(1, 2)), 1)
stay_check = pd.DataFrame([{'patients with measured vital-sign hours': int((obs.sum(axis=(1, 2)) > 0).sum()),
                            'median share equal to [15] (%)': round(100 * float(np.median(agree)), 1),
                            'patients below 50% equal': int((agree < 0.5).sum()),
                            'patients below 80% equal': int((agree < 0.8).sum())}])
stay_check.to_csv(f'{TAB_R}/stay_check_vitals.csv', index=False)
print('\nchosen stay against the cohort table (vital signs, measured hours):')
print(stay_check.to_string(index=False))
U_old = dyn_b[:, :, DYN.index(URINE)]
print(f'\nurine output ({URINE_SOURCE}): hours equal to [15] '
      f'{100 * np.mean(np.isclose(U_new, U_old, rtol=1e-3, atol=1e-3)):.1f}%')


## 7. Derived channels and the cutoff test

All channels are computed from the aligned hourly inputs, and all are causal: the channel at hour *t* uses hours 0 to *t* only.

| Channel | Definition |
|---|---|
| x | value, carried forward |
| Δx | hourly change of the carried-forward value; zero until the first observation |
| m | observation mask |
| δ | hours since the last observation: zero at an observed hour, 24 before the first observation, capped at 24 (GRU-D uses the convention of Che et al.) |
| d | change between the last two observed values, held until the next observation; zero until the second observation |
| c | cumulative number of hours with a measurement |

The three nearly constant laboratory inputs of the static pathway (thrombin, fibrinogen, albumin) take their first valid result in hours 0 to 23. Patients never tested are left missing and filled per fold with the median of the training patients.

**Cutoff test.** Every hour after a cutoff (hours 5, 11, and 17) is edited at random, and the test checks that no channel at or before the cutoff changes. A training window that ends at the cutoff sees only those hours, so its inputs do not depend on later records.

In [ ]:
# ============================================================
# 7. Representation components and the cutoff test
# ============================================================
comp = pa.components(V, M, U_new[:, :, None].astype(np.float32))
CHANNELS = {name: sum(1 if g in ('u', 'du') else len(MASKED) for g in groups)
            for name, groups in pa.REPRESENTATIONS.items()}
CHANNELS['GRUD'] = 3 * (len(MASKED) + 1)
print('channels per hour:', CHANNELS)

stat_num = stat_num0.copy()
for f, vals in static_first.items():
    stat_num[:, NUM.index(f)] = vals          # first result in hours 0-23; missing stays missing
print('static laboratory inputs rebuilt:', list(static_first) or 'none',
      '| never measured in hours 0-23 (%):',
      {f: round(100 * float(np.isnan(v).mean()), 1) for f, v in static_first.items()})

cut = pa.causality_test(V, M, comp['U'])
cut.to_csv(f'{TAB_R}/cutoff_test.csv', index=False)
print(cut.pivot(index='channel', columns='hours compared',
                values='unchanged after editing later hours').to_string())
assert cut['unchanged after editing later hours'].all(), 'a channel depends on later hours'
print('cutoff test passed: no channel at or before an hour depends on later records')


## 8. Partitions

* **Cross-validation and test.** Every patient keeps the partition of [15]: the held-out test set or fold *k*.
* **Early stopping in cross-validation.** Each fold sets aside 15% of its training patients (stratified by outcome, `random_state = 100 + k`, the same for every model and seed) as an internal split. The evaluated fold is never monitored.
* **Test.** 85% of the development patients train the model, and 15% serve for early stopping and Platt scaling (`random_state = 7`).
* **Temporal.** Patients with `anchor_year_group` 2008–2016 form the development set, split 85/15 in the same way, and those with 2017–2019 the evaluation set.

Each model is trained in its own short-lived Python process: TensorFlow keeps every traced training graph for the life of a process, which would exhaust the memory of a Colab runtime over the hundreds of fits that follow.

In [ ]:
# ============================================================
# 8. Partitions
# ============================================================
from sklearn.model_selection import train_test_split

fold_of = (pd.Series(folds['fold'].to_numpy(), index=folds[ID].to_numpy().astype(pids.dtype))
             .reindex(pids).to_numpy(dtype=float))
is_test = np.isin(pids, test_ids)
dev = np.flatnonzero(~is_test & ~np.isnan(fold_of))      # all patients of [15] are analysed
test = np.flatnonzero(is_test)


def stop_split(rows, seed):
    a, b = train_test_split(rows, test_size=STOP_FRAC, stratify=y_all[rows], random_state=seed)
    return np.sort(a), np.sort(b)


CV_SPLIT = {}
for k in sorted(np.unique(fold_of[dev]).astype(int)):
    fit, stop = stop_split(dev[fold_of[dev] != k], 100 + k)
    CV_SPLIT[k] = (fit, stop, dev[fold_of[dev] == k])
TEST_SPLIT = (*stop_split(dev, 7), test)
tdev = np.flatnonzero(PERIOD < 2017)
TEMP_SPLIT = (*stop_split(tdev, 7), np.flatnonzero(PERIOD >= 2017))

rows_ = []
for name, (a, b, c) in ([(f'CV fold {k}', v) for k, v in CV_SPLIT.items()]
                        + [('test protocol', TEST_SPLIT), ('temporal protocol', TEMP_SPLIT)]):
    for part, r in (('fit', a), ('early stopping', b), ('evaluated', c)):
        rows_.append({'design': name, 'part': part, 'patients': len(r),
                      'deaths': int(y_all[r].sum()), 'mortality (%)': round(100 * y_all[r].mean(), 2)})
parts_tab = pd.DataFrame(rows_)
parts_tab.to_csv(f'{TAB_R}/partition_summary.csv', index=False)
print(parts_tab.to_string(index=False))

assign = pd.Series('excluded', index=pids, dtype=object)
for k, (_, _, ev) in CV_SPLIT.items():
    assign.iloc[ev] = f'fold {k}'
assign.iloc[test] = 'test'
pd.DataFrame({ID: pids, 'assignment': assign.values, 'period': PERIOD}).to_csv(
    f'{CACHE_R}/partition_rev.csv', index=False)       # patient level: stays on Drive
assert (assign.values == SPLIT).all(), 'partition differs from the training table'

# Each model is trained in its own short-lived Python process (isolate=True):
# TensorFlow keeps every traced training graph for the life of a process,
# about 70-100 MB per model, which would exhaust the RAM of a Colab runtime
# over the hundreds of fits below. Inputs pass through a local folder.
ctx = pa.Context(y_all, pids, W, S, wn.make_windows, PRED_R, LOG, CV_SPLIT, TEST_SPLIT,
                 TEMP_SPLIT, EPOCHS, BATCH, PATIENCE, DEVICE, src_dir=SRC, train_module=tr,
                 isolate=True, work_dir='/tmp/paperA_rev_work')
print('\nready to train on', DEVICE)


## 9. Audits: winsorisation, static inputs, and the timing of the predictors

* **(a) Winsorisation.** For each set of patients whose statistics are fitted (the five cross-validation folds, the test protocol, and the temporal protocol), every channel and variable is checked for its 1st and 99th percentile limits, whether it becomes constant, and the share of training cells clipped. Two rules are compared:
  * **rule of [15]:** percentiles over all training cells;
  * **revised rule:** limits of the change channels (Δx, d) from their nonzero cells, and no winsorisation of the channels bounded by construction (m, δ, c).

  For a sparse change channel, such as Δx of lactate, most training cells are zero, so percentile limits over all cells can make the channel constant. The models trained here therefore use the revised rule; R2_old and R3_old keep the rule of [15]. A channel whose training standard deviation is zero is divided by one and becomes a constant zero.
* **(b) The static inputs of [15]**, with their encoding and time window. The SOFA score at Sepsis-3 onset is listed but not used, because onset can lie after the landmark or in another ICU stay; every model receives the other 23 inputs.
* **(c) Timing of each dynamic source** relative to the landmark.

In [ ]:
# ============================================================
# 9. Audits: winsorisation, static inputs, timing of the predictors
# ============================================================
FIT_SETS = ({f'CV fold {k}': v[0] for k, v in CV_SPLIT.items()}
            | {'test protocol': TEST_SPLIT[0], 'temporal protocol': TEMP_SPLIT[0]})
VARS = {'x': MASKED, 'u': [URINE], 'dx': MASKED, 'du': [URINE], 'm': MASKED,
        'dt': MASKED, 'd': MASKED, 'c': MASKED, 'x (GRU-D, observed values)': MASKED}
SRC_OF = {'u': 'U', 'dx': 'DX', 'du': 'DU', 'm': 'M', 'dt': 'DT', 'd': 'D', 'c': 'C'}


def constant(p, j):
    return bool(p['constant'][j] or p['lo'][j] == p['hi'][j])


rows_ = []
for set_name, fit in FIT_SETS.items():
    old = pa.channel_groups(comp, fit, set(SRC_OF) | {'x'}, rule='paper15')[1]
    new = pa.channel_groups(comp, fit, set(SRC_OF) | {'x'}, rule='revised')[1]
    old['x (GRU-D, observed values)'] = new['x (GRU-D, observed values)'] = pa.grud_inputs(comp, fit)[1]
    for g in new:
        raw = comp[SRC_OF[g]][fit] if g in SRC_OF else None
        for j, var in enumerate(VARS[g]):
            rows_.append({
                'fit set': set_name, 'channel': g, 'variable': var,
                'nonzero training cells (%)': 100 * float(np.mean(raw[:, :, j] != 0)) if raw is not None else np.nan,
                'limits, rule of [15]': f"{old[g]['lo'][j]:.4g} to {old[g]['hi'][j]:.4g}",
                'constant, rule of [15]': constant(old[g], j),
                'clipped (%), rule of [15]': 100 * old[g]['clipped_share'][j],
                'limits, revised rule': f"{new[g]['lo'][j]:.4g} to {new[g]['hi'][j]:.4g}",
                'constant, revised rule': constant(new[g], j),
                'clipped (%), revised rule': 100 * new[g]['clipped_share'][j]})
win = pd.DataFrame(rows_)
win.to_csv(f'{TAB_R}/winsorisation_check.csv', index=False)
print(f'(a) {len(win):,} combinations of fit set, channel and variable')
for rule in ('rule of [15]', 'revised rule'):
    flag = win[win[f'constant, {rule}']]
    print(f'    constant after winsorisation, {rule}: {len(flag)}')
    if len(flag):
        print(flag[['fit set', 'channel', 'variable', 'nonzero training cells (%)',
                    f'limits, {rule}', f'clipped (%), {rule}']].round(2).to_string(index=False))
mrow = win[win.channel == 'm']
print(f"    mask channels keeping both values under the rule of [15] (limits 0 and 1): "
      f"{int((mrow['limits, rule of [15]'] == '0 to 1').sum())} of {len(mrow)}; "
      f"the revised rule does not winsorise the mask, time since observation, or counts")
print('    zero-variance channels are divided by one and become a constant zero')

# (b) The static inputs of [15]: 23 are used, and the SOFA score at onset is
# listed but not used. Descriptions follow the extraction queries of [15];
# the first matching key in this order is used.
STATIC_INFO = [
    ('onset', 'SOFA score at Sepsis-3 onset', 'sepsis3', 'not used: can lie after the landmark or in another ICU stay'),
    ('respiration', 'SOFA respiration component, worst over hours 0-23', 'sofa (hourly)', 'rolling 24-h windows ending in hours 0-23'),
    ('coagulation', 'SOFA coagulation component, worst over hours 0-23', 'sofa (hourly)', 'rolling 24-h windows ending in hours 0-23'),
    ('liver', 'SOFA liver component, worst over hours 0-23', 'sofa (hourly)', 'rolling 24-h windows ending in hours 0-23'),
    ('cardiovascular', 'SOFA cardiovascular component, worst over hours 0-23', 'sofa (hourly)', 'rolling 24-h windows ending in hours 0-23'),
    ('cns', 'SOFA central nervous system component, worst over hours 0-23', 'sofa (hourly)', 'rolling 24-h windows ending in hours 0-23'),
    ('renal', 'SOFA renal component, worst over hours 0-23', 'sofa (hourly)', 'rolling 24-h windows ending in hours 0-23'),
    ('sofa', 'First-day SOFA score, maximum rolling 24-h value over hours 0-23', 'sofa (hourly)',
     'windows ending in hours 0-23; each looks back 24 h, so it can include records from before ICU admission'),
    ('charlson', 'Charlson comorbidity index', 'diagnoses_icd of the index admission',
     'codes assigned at hospital discharge: not available in real time at the landmark (sensitivity analysis without it, Cell 15)'),
    ('gcs', 'Minimum Glasgow Coma Scale over hours 0-23', 'sofa (hourly gcs_min)', 'hours 0-23'),
    ('vaso', 'Vasopressor given in hours 0-23', 'sofa (hourly vasopressor rates)', 'hours 0-23'),
    ('vent', 'Invasive ventilation started within 24 h of ICU admission', 'ventilation',
     'start before ICU admission + 24 h (up to 59 min past the clock-hour landmark)'),
    ('thrombin', 'Thrombin time, first valid result in hours 0-23', 'laboratory extract (rebuilt)', 'hours 0-23'),
    ('fibrinogen', 'Fibrinogen, first valid result in hours 0-23', 'laboratory extract (rebuilt)', 'hours 0-23'),
    ('albumin', 'Albumin, first valid result in hours 0-23', 'laboratory extract (rebuilt)', 'hours 0-23'),
    ('age', 'Age at ICU admission', 'patients, icustays', 'at admission'),
]


def describe(col):
    low = col.lower()
    for key, *info in STATIC_INFO:
        if key in low:
            return info
    return ['(describe manually)', '', '']


E = np.arange(N)
rows_ = []
for j, c in enumerate(NUM):
    desc, src, window = describe(c)
    rows_.append({'input': c, 'encoding': 'numeric; training median where missing; standardised',
                  'description': desc, 'source': src, 'time window': window,
                  'missing before fill (%)': round(100 * float(np.isnan(stat_num[E, j]).mean()), 2)})
for lvl, col in zip(CAT_LEVELS, stat_cat.T):
    rows_.append({'input': lvl, 'encoding': 'one-hot; standardised',
                  'description': 'sex' if lvl.lower().startswith(('gender', 'sex'))
                                 else 'race or ethnicity, collapsed to six groups in [15]',
                  'source': 'patients, admissions', 'time window': 'at admission',
                  'patients (%)': round(100 * float(col[E].mean()), 1)})
static_tab = pd.DataFrame(rows_)
static_tab.to_csv(f'{TAB_R}/static_inputs.csv', index=False)
n_used = len(NUM) + stat_cat.shape[1] - sum('onset' in c.lower() for c in NUM)
print(f'\n(b) {len(NUM)} numeric + {stat_cat.shape[1]} one-hot = '
      f'{len(NUM) + stat_cat.shape[1]} static inputs of [15]; {n_used} are used')
print(static_tab[['input', 'description', 'time window']].to_string(index=False))

# (c) Timing of the dynamic sources (written when the grid was built).
p_audit = f'{TAB_R}/source_audit.csv'
if os.path.exists(p_audit):
    a = pd.read_csv(p_audit)
    a['records in hour 0 before ICU admission (%)'] = (
        100 * a['records in hour 0 before ICU admission'] / a['records in hours 0-23']).round(2)
    print('\n(c)', a[['source', 'records in hours 0-23', 'records in hour 0 before ICU admission (%)',
                      'records at or after the landmark kept']].to_string(index=False))


## 10. Optional: reproduction of the training of [15]

This cell retrains the reference configuration of [15] with its own code and protocol (seed 42, five folds) and compares the mean AUROC with the value of [15] in `results_rebuilt/e1_window_grid.csv`; the difference must be below 0.004. It confirms that the TensorFlow and Keras versions of the current runtime behave as before. It needs to run only once; the result is saved.

In [ ]:
# ============================================================
# 10. Optional: this environment reproduces the training of [15]
# ============================================================
RUN_GATE = True
GATE_FILE = f'{RES_R}/gate.json'
if os.path.exists(GATE_FILE):
    print('gate passed earlier:', json.load(open(GATE_FILE)))
elif RUN_GATE:
    e1 = pd.read_csv(f'{RES_B}/e1_window_grid.csv')
    ref = float(e1[(e1.window_size == W) & (e1.step == S) & (e1.seed == 42)].auroc_mean.iloc[0])
    aucs = []
    for k in sorted(folds.fold.unique()):
        va = np.array([row[p] for p in folds.loc[folds.fold == k, ID] if p in row])
        trn = np.array([row[p] for p in folds.loc[folds.fold != k, ID] if p in row])
        m, _, _ = tr.train_fold(dyn_b, stat_b, y_all, trn, va, window_size=W, step=S,
                                hp=BEST_HP, variant='full', seed=42, epochs=40,
                                batch_size=512, n_hours=24)
        aucs.append(m['auroc'])
        print(f'  fold {k}: AUROC {m["auroc"]:.4f}')
    got = float(np.mean(aucs))
    print(f'reproduced {got:.4f} against {ref:.4f} in [15] (difference {got - ref:+.4f})')
    assert abs(got - ref) < 0.004, 'this environment does not reproduce the training of [15]; stop here'
    json.dump({'reproduced': got, 'reported in [15]': ref, 'tensorflow': tf.__version__,
               'keras': keras.__version__}, open(GATE_FILE, 'w'), indent=2)


## 11. Model definitions

The channel-based models use the architecture and hyperparameters of [15] unchanged (`train.build_model`):

| Model | Inputs |
|---|---|
| R1 | values |
| R2 | values + Δx (reference) |
| R3 | values + m |
| R4 | R3 + δ |
| R5 | R4 + d |
| T | m + δ (timing only) |
| C | values + cumulative counts |
| S | static inputs only |

S keeps the static pathway and output layer of the other models.

The **GRU-D family** uses the setting selected in Cell 13:

| Model | Content |
|---|---|
| GRUD | GRU-D with the time gap of Che et al.; the last observation as x_last and the mean of the observed training values as x̄ |
| GRUD_reset | the same, with the δ channel of R4, as in the first submission |
| GRUS | one-directional GRU without decay, inputs [x, m, δ] |
| BIGRUS | the same, bidirectional, with the units split between the two directions |

**TBAL** is a bidirectional LSTM with time-aware attention on the R4 channels, with the setting selected in Cell 13.

**Earlier inputs and sensitivity analyses** (Cell 15):

| Model | Content |
|---|---|
| R2_old | the values of the cohort table of [15], carried forward and filled backward |
| R3_old | R2_old + the mask of the first submission |
| R1_noCCI, R2_noCCI | R1 and R2 without the Charlson comorbidity index |

Every model receives the same 23 static inputs; the SOFA score at Sepsis-3 onset is not used.

In [ ]:
# ============================================================
# 11. Model definitions (no training here)
# ============================================================
SPACE = {'units': [32, 64, 128], 'learning_rate': [1e-3, 1e-4, 5e-5],
         'l2': [1e-2, 5e-3, 1e-3], 'dropout': [0.2, 0.3, 0.4]}
CHARLSON = [c for c in NUM if 'charlson' in c.lower()]
ZERO_DYN = np.zeros((N, 24, 1), np.float32)      # placeholder windows for the static-only model


# The SOFA score at sepsis onset of [15] is not used: onset can lie after the
# landmark or in another ICU stay. Every model receives the other 23 inputs.
NOT_USED = [c for c in NUM if 'onset' in c.lower()]


def static_for(fit, drop=()):
    keep = [j for j, c in enumerate(NUM) if c not in drop and c not in NOT_USED]
    return pa.static_inputs(stat_num[:, keep], stat_cat, fit)


def channel_spec(name, drop=()):
    groups = pa.REPRESENTATIONS[name]

    def prepare(fit):
        g, _ = pa.channel_groups(comp, fit, set(groups))
        return pa.assemble(g, groups), static_for(fit, drop)
    return {'prepare': prepare, 'recipe': {'kind': 'cnn_bigru', 'hp': BEST_HP}}


def grud_family_spec(kind, hp, convention='che'):
    def prepare(fit):
        return pa.grud_inputs(comp, fit, convention)[0], static_for(fit)
    return {'prepare': prepare,
            'recipe': {'kind': {'GRUD': 'grud', 'GRUS': 'gru', 'BIGRUS': 'bigru'}[kind], 'hp': hp}}


def tbal_spec(hp):
    groups = pa.REPRESENTATIONS['TBAL']

    def prepare(fit):
        g, _ = pa.channel_groups(comp, fit, set(groups))
        return pa.assemble(g, groups), static_for(fit)
    return {'prepare': prepare, 'recipe': {'kind': 'tbal', 'hp': hp}}


def old_spec(name):
    def prepare(fit):
        return pa.old_inputs(name, dyn_b, mask_old, fit, N_DYN)[0], static_for(fit)
    return {'prepare': prepare, 'recipe': {'kind': 'cnn_bigru', 'hp': BEST_HP}}


SPECS = {name: channel_spec(name) for name in ('R1', 'R2', 'R3', 'R4', 'R5', 'T', 'C')}
SPECS['S'] = {'prepare': lambda fit: (ZERO_DYN, static_for(fit)),
              'recipe': {'kind': 'static', 'hp': BEST_HP}}
SPECS['R2_old'] = old_spec('R2_old')
if mask_old is not None:
    SPECS['R3_old'] = old_spec('R3_old')
if CHARLSON:
    SPECS['R1_noCCI'] = channel_spec('R1', drop=CHARLSON)
    SPECS['R2_noCCI'] = channel_spec('R2', drop=CHARLSON)

SELECTED = {}
for name in ('GRUD', 'TBAL'):
    p_sel = f'{RES_R}/selected_hp_{name}.json'
    if os.path.exists(p_sel):
        SELECTED[name] = json.load(open(p_sel))


def add_selected():
    """Register the models that need the hyperparameters of Cell 13."""
    if 'GRUD' in SELECTED:
        hp = SELECTED['GRUD']
        SPECS['GRUD'] = grud_family_spec('GRUD', hp, 'che')
        SPECS['GRUD_reset'] = grud_family_spec('GRUD', hp, 'reset')
        SPECS['GRUS'] = grud_family_spec('GRUS', hp)
        SPECS['BIGRUS'] = grud_family_spec('BIGRUS', hp)
    if 'TBAL' in SELECTED:
        SPECS['TBAL'] = tbal_spec(SELECTED['TBAL'])


def run(names, design='TEST'):
    """Cross-validation and the held-out protocol for each model and seed;
    finished folds and runs are skipped, so this resumes."""
    for name in names:
        for seed in SEEDS:
            t0 = time.time()
            pa.run_folds(name, SPECS[name], ctx, seed)
            pa.run_holdout(name, SPECS[name], ctx, seed, design)
            print(f'{name} seed {seed}: complete ({time.time() - t0:.0f}s this session)')


add_selected()
print('static inputs per model:', len(NUM) - len(NOT_USED) + stat_cat.shape[1])
print('defined:', sorted(SPECS))
print('hyperparameters selected:', SELECTED or 'none yet (Cell 13)')


## 12. Channel-based models: R1 to R5, T, C, and S

Three seeds, each with five cross-validation folds and one held-out test run. The order can be changed in `MAIN_MODELS`; every fold is saved as soon as it finishes.

In [ ]:
# ============================================================
# 12. Train the channel models
# ============================================================
MAIN_MODELS = ['R2', 'R1', 'R3', 'R4', 'R5', 'T', 'C', 'S']
run(MAIN_MODELS)
print('channel models complete')


## 13. Random search for GRU-D and TBAL

The search budget of [15]: 15 distinct draws from a discrete grid (sampling seed 2026), one training seed (42), three development folds (1 to 3), and the mean AUROC as the objective, with early stopping on the internal split. The grid is units {32, 64, 128}, learning rate {1e-3, 1e-4, 5e-5}, L2 {1e-2, 5e-3, 1e-3}, and dropout {0.2, 0.3, 0.4} (`config/search_space.json`). GRUS and BIGRUS use the setting selected for GRU-D, so that their comparison with it is matched.

In [ ]:
# ============================================================
# 13. Hyperparameter search for GRU-D and TBAL
# ============================================================
for name, make in (('GRUD', lambda hp: grud_family_spec('GRUD', hp, 'che')), ('TBAL', tbal_spec)):
    p_sel = f'{RES_R}/selected_hp_{name}.json'
    if os.path.exists(p_sel):
        SELECTED[name] = json.load(open(p_sel))
        print(name, 'selected earlier:', SELECTED[name])
        continue
    res, best = pa.run_search(name, make, ctx, SPACE, f'{TAB_R}/search_{name}.csv', n_trials=N_TRIALS)
    SELECTED[name] = best
    json.dump(best, open(p_sel, 'w'), indent=2)
    print(res.sort_values('mean AUROC', ascending=False).round(4).to_string(index=False))
    print(name, 'selected:', best)
add_selected()


## 14. GRU-D family and TBAL

The sequence GRUD, GRUS, BIGRUS, R4 separates the sources of the difference between GRU-D and R4:

* GRUD against GRUS: the **decay**;
* BIGRUS against GRUS: the **direction**;
* R4 against BIGRUS: the **convolutional block**.

GRUD against GRUD_reset tests the **convention for the time gap**, with identical data and optimisation.

In [ ]:
# ============================================================
# 14. Train the GRU-D family and TBAL
# ============================================================
assert {'GRUD', 'TBAL'} <= set(SELECTED), 'run Cell 13 first'
run(['GRUD', 'GRUD_reset', 'GRUS', 'BIGRUS', 'TBAL'])
print('GRU-D family and TBAL complete')


## 15. Earlier inputs, and models without the Charlson index

R2_old and R3_old are trained with the same protocol, patients, and static inputs as the other models. Only their dynamic inputs differ: the values of the cohort table of [15] and, for R3_old, the mask of the first submission, which was reconstructed separately. This separates the effect of how the inputs are assembled from that of the representation.

R1_noCCI and R2_noCCI remove the Charlson comorbidity index, whose codes are assigned at hospital discharge.

In [ ]:
# ============================================================
# 15. Existing pipeline against the rebuilt one, and no Charlson
# ============================================================
EXTRA = [m for m in ('R2_old', 'R3_old', 'R1_noCCI', 'R2_noCCI') if m in SPECS]
run(EXTRA)
print('complete:', EXTRA)


## 16. Temporal validation: fix the analysis plan first

The plan lists the models, seeds, contrasts, the margin of ±0.01, the operating points, and the bootstrap settings. The plan of the paper is `config/analysis_plan_temporal.json`, and its SHA-256, recorded before any temporal model was trained, is in `config/analysis_plan_temporal.sha256`. When that file is present, this cell uses it unchanged, so the printed hash can be checked against the recorded one; otherwise it writes a new plan. Cell 17 stores the same hash with its results.

In [ ]:
# ============================================================
# 16. Temporal validation: fix the analysis plan first
# ============================================================
import shutil

PLAN = f'{TAB_R}/analysis_plan_temporal.json'
PLAN_REPO = f'{BASE}/config/analysis_plan_temporal.json'   # the plan of the paper
assert 'GRUD' in SELECTED, 'run Cell 13 first'
if not os.path.exists(PLAN) and os.path.exists(PLAN_REPO):
    shutil.copy(PLAN_REPO, PLAN)        # fixed before training; its SHA-256 is in config/
if not os.path.exists(PLAN):
    plan = {
        'written': time.strftime('%Y-%m-%d %H:%M:%S'),
        'question': 'Do the paired differences from R2 and the operating-point conclusions '
                    'hold in a later calendar period?',
        'cohort': f'all {N:,} patients of [15]',
        'development': 'anchor_year_group 2008 - 2010, 2011 - 2013, 2014 - 2016; 85% for '
                       'training, 15% for early stopping and Platt scaling',
        'evaluation': 'anchor_year_group 2017 - 2019, evaluated once',
        'models': ['R2', 'R1', 'R3', 'R4', 'R5', 'GRUD'], 'seeds': SEEDS, 'reference': 'R2',
        'primary contrasts': ['R1 - R2', 'R3 - R2', 'R4 - R2', 'R5 - R2'],
        'secondary contrasts': ['GRUD - R2'],
        'tolerance': {'metric': 'AUROC', 'margin': MARGIN,
                      'rule': 'difference bounded by the margin when the 90% DeLong interval '
                              'lies inside +/- margin'},
        'adjustment': 'Benjamini-Hochberg within the primary contrasts',
        'operating points': [p[0] for p in pa.POINTS],
        'bootstrap': {'resamples': 2000, 'seed': 2026},
        'hyperparameters': {'CNN-BiGRU': BEST_HP, 'GRUD': SELECTED['GRUD']},
    }
    json.dump(plan, open(PLAN, 'w'), indent=2, default=str)
print(open(PLAN).read())
print('SHA-256:', pa.sha256(PLAN))
print('\nKeep this file unchanged until Cell 17 has finished.')


## 17. Temporal validation runs

The period 2017–2019 is evaluated once, with exactly the models, seeds, and contrasts of the plan.

In [ ]:
# ============================================================
# 17. Temporal validation runs
# ============================================================
plan = json.load(open(PLAN))
for name in plan['models']:
    for seed in plan['seeds']:
        pa.run_holdout(name, SPECS[name], ctx, seed, design='TEMP')
    print(name, 'temporal: complete')
json.dump({'plan sha256': pa.sha256(PLAN), 'completed': time.strftime('%Y-%m-%d %H:%M:%S')},
          open(f'{RES_R}/temporal_completed.json', 'w'), indent=2)


## 18. Performance and calibration

Probabilities are averaged across seeds before any metric is computed, as in [15]. The AUPRC is the average precision: the sum of precisions weighted by the increase in recall, without interpolation. The same estimator is used for the point estimate and for every bootstrap resample.

Calibration is reported as the slope and intercept of logistic recalibration and as calibration-in-the-large (the intercept with the slope fixed at one), with bootstrap 95% intervals for the test and temporal sets. Cross-validation predictions are not recalibrated; the models are trained with class weights.

In [ ]:
# ============================================================
# 18. Performance with intervals
# ============================================================
ALL = ['R1', 'R2', 'R3', 'R4', 'R5', 'T', 'C', 'S', 'GRUD', 'GRUD_reset', 'GRUS', 'BIGRUS',
       'TBAL', 'R2_old', 'R3_old', 'R1_noCCI', 'R2_noCCI']
PATTERN = {'CV': 'CV_{m}_s{{seed}}.npz', 'TEST': 'TEST_{m}_platt_s{{seed}}.npz',
           'TEMP': 'TEMP_{m}_platt_s{{seed}}.npz'}
POOL = {d: {m: pa.pooled(PRED_R, pat.format(m=m), SEEDS) for m in ALL
            if pa.available(PRED_R, pat.format(m=m), SEEDS)} for d, pat in PATTERN.items()}
for d, pool in POOL.items():
    print(f'{d}: {list(pool)}')

perf = pd.DataFrame([{'design': d, 'model': m, **pa.performance(t.y.values, t.score.values, ci=(d != 'CV'))}
                     for d, pool in POOL.items() for m, t in pool.items()])
perf.to_csv(f'{TAB_R}/performance.csv', index=False)
show = ['design', 'model', 'n', 'AUROC', 'AUPRC', 'Brier', 'calibration slope',
        'calibration-in-the-large']
print(perf[show].round(4).to_string(index=False))
ci_cols = [c for c in perf.columns if c.endswith('95% CI')]
if ci_cols:
    print()
    print(perf[perf.design != 'CV'][['design', 'model'] + ci_cols].to_string(index=False))


## 19. Paired comparisons and bounded differences

AUROC differences are tested with the DeLong test; linear contrasts, such as the factorial interaction, use the DeLong covariance. AUPRC differences use 2,000 paired bootstrap resamples.

P values are adjusted with the Benjamini–Hochberg procedure within each family. Every contrast is also reported with its 90% interval and two one-sided tests at ±0.01: a difference is *bounded* within the margin, in this cohort, when its 90% interval lies inside it.

For the temporal set, the families follow the plan fixed in Cell 16.

In [ ]:
# ============================================================
# 19. Paired comparisons and bounded differences
# ============================================================
def c(a, b):
    return {a: 1, b: -1}


FAMILIES = {
    'representations against R2': [('R1 - R2', c('R1', 'R2')), ('R3 - R2', c('R3', 'R2')),
                                   ('R4 - R2', c('R4', 'R2')), ('R5 - R2', c('R5', 'R2')),
                                   ('R4 - R3', c('R4', 'R3')), ('R5 - R4', c('R5', 'R4'))],
    'values and timing': [('R1 - S: values', c('R1', 'S')), ('T - S: timing', c('T', 'S')),
                          ('R4 - R1: timing given values', c('R4', 'R1')),
                          ('R4 - T: values given timing', c('R4', 'T')),
                          ('interaction (R4 - T) - (R1 - S)', {'R4': 1, 'T': -1, 'R1': -1, 'S': 1}),
                          ('C - R1: cumulative counts', c('C', 'R1')), ('C - R3', c('C', 'R3'))],
    'irregularity inside the model': [('GRUD - R1', c('GRUD', 'R1')), ('GRUD - R2', c('GRUD', 'R2')),
                                      ('GRUD - R4', c('GRUD', 'R4')), ('TBAL - R1', c('TBAL', 'R1')),
                                      ('TBAL - R2', c('TBAL', 'R2')), ('TBAL - R4', c('TBAL', 'R4'))],
    'GRU-D decomposition': [('GRUD - GRUD_reset: gap convention', c('GRUD', 'GRUD_reset')),
                            ('GRUD - GRUS: decay', c('GRUD', 'GRUS')),
                            ('BIGRUS - GRUS: direction', c('BIGRUS', 'GRUS')),
                            ('R4 - BIGRUS: convolutional block', c('R4', 'BIGRUS'))],
    'pipeline and predictors': [('R2 - R2_old: rebuilt inputs', c('R2', 'R2_old')),
                                ('R3 - R3_old: aligned mask', c('R3', 'R3_old')),
                                ('R1_noCCI - R1: without Charlson', c('R1_noCCI', 'R1')),
                                ('R2_noCCI - R2: without Charlson', c('R2_noCCI', 'R2'))],
}
TEMP_FAMILIES = {'primary (plan)': FAMILIES['representations against R2'][:4],
                 'secondary (plan)': [('GRUD - R2', c('GRUD', 'R2'))]}


def compare(design, pool, families):
    out = []
    for fam, contrasts in families.items():
        recs = []
        for label, w in contrasts:
            if design != 'CV' and label in ('R4 - R3', 'R5 - R4'):
                continue                      # as in the first submission: CV only
            if not all(m in pool for m in w):
                continue
            common = sorted(set.intersection(*(set(pool[m].index) for m in w)))
            y = pool[next(iter(w))].loc[common, 'y'].to_numpy()
            sc = {m: pool[m].loc[common, 'score'].to_numpy() for m in w}
            a = pa.auc_contrast(y, sc, w, MARGIN)
            ap, lo, hi = pa.ap_contrast(y, sc, w)
            recs.append({'design': design, 'family': fam, 'contrast': label, 'n': len(common),
                         'AUROC difference': a['estimate'],
                         '95% CI': f"{a['95% low']:+.4f} to {a['95% high']:+.4f}", 'p': a['p'],
                         '90% CI': f"{a['90% low']:+.4f} to {a['90% high']:+.4f}",
                         'bounded by 0.01 (90% CI)': bool(a['90% low'] > -MARGIN and a['90% high'] < MARGIN),
                         'TOST p': a['tost p'], 'AUPRC difference': ap,
                         'AUPRC 95% CI': f'{lo:+.4f} to {hi:+.4f}'})
        for r, q in zip(recs, pa.benjamini_hochberg([r['p'] for r in recs])):
            r['adjusted p'] = q
        out += recs
    return out


cmp_ = pd.DataFrame(compare('CV', POOL['CV'], FAMILIES) + compare('TEST', POOL['TEST'], FAMILIES)
                    + compare('TEMP', POOL['TEMP'], TEMP_FAMILIES))
cmp_.to_csv(f'{TAB_R}/comparisons.csv', index=False)
show = ['design', 'family', 'contrast', 'AUROC difference', '95% CI', 'adjusted p', '90% CI',
        'bounded by 0.01 (90% CI)', 'AUPRC difference', 'AUPRC 95% CI']
for d in ('CV', 'TEST', 'TEMP'):
    sub = cmp_[cmp_.design == d]
    if len(sub):
        print(f'\n[{d}]')
        print(sub[show].round(4).to_string(index=False))


## 20. Operating points and net benefit

The decision consequences of the AUROC differences at the 24-hour landmark. The operating points are the top 10% and top 20% of patients by predicted risk (an alert budget) and the risk thresholds 0.15, 0.30, and 0.50.

Net benefit weighs false positives by the odds of the risk threshold *p*ₜ, so it is defined for the risk thresholds only; a budget cutoff is reported with sensitivity and positive predictive value. All differences from R2 use 2,000 paired bootstrap resamples.

In [ ]:
# ============================================================
# 20. Operating points, net benefit, and decision curves
# ============================================================
OP_MODELS = ['R2', 'R1', 'R3', 'R4', 'R5', 'GRUD', 'TBAL']
for design in ('TEST', 'TEMP'):
    pool = POOL[design]
    ms = [m for m in OP_MODELS if m in pool]
    if 'R2' not in ms or len(ms) < 2:
        print(f'{design}: skipped (predictions missing)')
        continue
    common = sorted(set.intersection(*(set(pool[m].index) for m in ms)))
    y = pool['R2'].loc[common, 'y'].to_numpy()
    P = {m: pool[m].loc[common, 'score'].to_numpy() for m in ms}
    each, diff = pa.operating_points(y, P)
    each.to_csv(f'{TAB_R}/operating_points_{design}.csv', index=False)
    diff.to_csv(f'{TAB_R}/operating_points_vs_R2_{design}.csv', index=False)
    dca = pa.net_benefit_curves(y, P)
    dca_diff = dca.sub(dca['R2'], axis=0).drop(columns='R2')
    dca_diff.to_csv(f'{TAB_R}/net_benefit_difference_vs_R2_{design}.csv')
    print(f'\n[{design}] {len(y):,} patients, {int(y.sum()):,} deaths')
    print(each.round(1).to_string(index=False))
    print(diff.round(2).to_string(index=False))
    print('largest |net benefit difference| from R2 per 1000, thresholds 0.05-0.50:')
    print(dca_diff.abs().max().round(1).to_string())


## 21. Measurement counts by outcome and observation frequency

The number of measured hours of each variable among survivors and non-survivors, and the observation frequency of each variable, computed from the rebuilt masks of all patients.

In [ ]:
# ============================================================
# 21. Measurement counts by outcome and observation frequency
# ============================================================
from scipy.stats import mannwhitneyu
from sklearn.metrics import roc_auc_score

E = np.arange(N)
counts = M[E].sum(axis=1)
dd = y_all[E] == 1
rows_ = []
for j, f in enumerate(MASKED):
    a, b = counts[~dd, j], counts[dd, j]
    rows_.append({'variable': f, 'survivors >= 2 (%)': 100 * (a >= 2).mean(),
                  'non-survivors >= 2 (%)': 100 * (b >= 2).mean(),
                  'median survivors': np.median(a), 'median non-survivors': np.median(b),
                  'p': mannwhitneyu(a, b).pvalue, 'AUROC of the count alone': roc_auc_score(dd, counts[:, j])})
sched = pd.DataFrame(rows_)
sched['adjusted p'] = pa.benjamini_hochberg(sched['p'].values)
sched.to_csv(f'{TAB_R}/measurement_by_outcome.csv', index=False)
print(sched.round(3).to_string(index=False))

freq = pd.DataFrame({'feature': MASKED,
                     'hours observed (%)': 100 * M[E].mean(axis=(0, 1)),
                     'patients with >=1 observation (%)': 100 * (M[E].sum(axis=1) > 0).mean(axis=0),
                     'patients with >=2 observations (%)': 100 * (M[E].sum(axis=1) > 1).mean(axis=0)})
freq.round(1).to_csv(f'{TAB_R}/observation_frequency.csv', index=False)
print()
print(freq.round(1).sort_values('hours observed (%)', ascending=False).to_string(index=False))


## 22. Figures

Observation frequency, the AUROC differences from R2, and the ROC, precision-recall, and calibration curves on the held-out test set. Times New Roman 10 pt at 600 dpi, 8.2 cm wide for one column and 17.2 cm for two, as in the IJIES template. Where Times New Roman cannot be installed, Liberation Serif, which has the same letter widths, is used.

In [ ]:
# ============================================================
# 22. Figures
# ============================================================
import shutil
import subprocess
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm
from matplotlib.patches import Patch
from sklearn.calibration import calibration_curve
from sklearn.metrics import precision_recall_curve, roc_auc_score, roc_curve


def times_font():
    have = {f.name for f in fm.fontManager.ttflist}
    if 'Times New Roman' not in have and os.path.exists('/content') and shutil.which('apt-get'):
        try:
            subprocess.run('apt-get -qq update && echo "ttf-mscorefonts-installer '
                           'msttcorefonts/accepted-mscorefonts-eula select true" | '
                           'debconf-set-selections && apt-get -qq install -y '
                           'ttf-mscorefonts-installer fonts-liberation', shell=True,
                           stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, timeout=300)
        except Exception:
            pass
        for path in glob.glob('/usr/share/fonts/truetype/**/*.ttf', recursive=True):
            try:
                fm.fontManager.addfont(path)
            except Exception:
                pass
        have = {f.name for f in fm.fontManager.ttflist}
    for name in ('Times New Roman', 'Liberation Serif'):
        if name in have:
            return name
    return 'DejaVu Serif'


FONT = times_font()
plt.rcParams.update({'font.family': 'serif', 'font.serif': [FONT], 'font.size': 10,
                     'axes.labelsize': 10, 'xtick.labelsize': 10, 'ytick.labelsize': 10,
                     'legend.fontsize': 9, 'legend.frameon': False, 'axes.spines.top': False,
                     'axes.spines.right': False, 'axes.linewidth': 0.8, 'pdf.fonttype': 42,
                     'mathtext.fontset': 'stix', 'savefig.bbox': 'standard'})
CM = 1 / 2.54
LABEL = {'GRUD': 'GRU-D'}
COLOR = {'R1': '#0072B2', 'R2': '#E69F00', 'R3': '#009E73', 'R4': '#D55E00', 'R5': '#CC79A7',
         'GRUD': '#56B4E9', 'TBAL': '#000000'}
STYLE = {'R1': '-', 'R2': '--', 'R3': '-.', 'R4': ':', 'R5': (0, (5, 1)),
         'GRUD': (0, (3, 1, 1, 1)), 'TBAL': (0, (1, 1))}
MARK = {'R1': 'o', 'R2': 's', 'R3': '^', 'R4': 'D', 'R5': 'v', 'GRUD': 'P', 'TBAL': 'X'}


def save(fig, name):
    for ext in ('png', 'pdf'):
        fig.savefig(f'{FIG_R}/{name}.{ext}', dpi=600)
    plt.show()
    plt.close(fig)


print('font used:', FONT)

# Figure 2: observation frequency (one column).
NAMES = {'heart_rate': 'Heart rate', 'sbp': 'Systolic BP', 'dbp': 'Diastolic BP',
         'mbp': 'Mean arterial pressure', 'resp_rate': 'Respiratory rate',
         'temperature': 'Temperature', 'spo2': 'SpO2', 'glucose': 'Glucose', 'bun': 'BUN',
         'creatinine': 'Creatinine', 'platelets': 'Platelets', 'bicarbonate': 'Bicarbonate',
         'aniongap': 'Anion gap', 'inr': 'INR', 'pt': 'PT', 'ptt': 'PTT',
         'wbc': 'White blood cells', 'calcium': 'Calcium', 'chloride': 'Chloride',
         'sodium': 'Sodium', 'potassium': 'Potassium', 'lactate': 'Lactate'}
fq = freq.sort_values('hours observed (%)')
fig, ax = plt.subplots(figsize=(8.2 * CM, 10 * CM), layout='constrained')
ax.barh([NAMES.get(f, f) for f in fq.feature], fq['hours observed (%)'],
        color=['#0072B2' if f in pa.VITAL else '#E69F00' for f in fq.feature],
        edgecolor='black', linewidth=0.4, height=0.72)
ax.set(xlim=(0, 100), xlabel='Hours recorded (%)')
ax.set_xticks(range(0, 101, 20))
ax.xaxis.grid(True, color='0.85', linewidth=0.6)
ax.set_axisbelow(True)
ax.margins(y=0.01)
ax.legend(handles=[Patch(facecolor='#0072B2', edgecolor='black', linewidth=0.4, label='Vital sign'),
                   Patch(facecolor='#E69F00', edgecolor='black', linewidth=0.4, label='Laboratory')],
          loc='lower right', frameon=True, facecolor='white', edgecolor='none', framealpha=1)
save(fig, 'fig2_observation_frequency')

# Figure 5: AUROC difference against R2 in each design (two columns).
# Thick bars: 90% CI, the interval behind the ±0.01 equivalence judgement (two one-sided
# tests); thin lines: 95% CI. Neither interval is adjusted for multiplicity.
order = [m for m in ('R1', 'R3', 'R4', 'R5', 'GRUD', 'TBAL')]
designs = [d for d in ('CV', 'TEST', 'TEMP') if (cmp_.design == d).any()]
titles = {'CV': 'Cross-validation', 'TEST': 'Held-out test', 'TEMP': 'Temporal validation'}


def interval(text):
    return [float(v) for v in text.split(' to ')]


fig, axes = plt.subplots(1, len(designs), figsize=(17.2 * CM, 6.2 * CM), sharex=True, sharey=True,
                         layout='constrained', squeeze=False)
for ax, d in zip(axes[0], designs):
    ypos = np.arange(len(order))[::-1]
    ax.axvline(0, color='0.4', lw=0.7, ls='--', zorder=1)
    for mv in (-MARGIN, MARGIN):
        ax.axvline(mv, color='0.55', lw=0.7, ls=':', zorder=1)
    for yy, m in zip(ypos, order):
        r = cmp_[(cmp_.design == d) & (cmp_.contrast == f'{m} - R2')]
        if not len(r):
            ax.text(0.5, yy, 'not in the temporal plan' if d == 'TEMP' else 'not evaluated',
                    transform=ax.get_yaxis_transform(), ha='center', va='center', fontsize=8,
                    color='0.4', style='italic', zorder=3,
                    bbox=dict(facecolor='white', edgecolor='none', pad=1))
            continue
        lo, hi = interval(r['95% CI'].iloc[0])
        ax.plot([lo, hi], [yy, yy], color=COLOR[m], lw=0.9, solid_capstyle='butt', zorder=2)
        lo, hi = interval(r['90% CI'].iloc[0])
        ax.plot([lo, hi], [yy, yy], color=COLOR[m], lw=3.2, solid_capstyle='butt', zorder=2)
        ax.plot(r['AUROC difference'].iloc[0], yy, MARK[m], color=COLOR[m], ms=5.5,
                mec='white', mew=0.7, zorder=3)
    ax.set_yticks(ypos)
    ax.set_yticklabels([LABEL.get(m, m) for m in order])
    ax.set_title(titles[d], fontsize=10)
fig.supxlabel('AUROC difference against R2 (thick: 90% CI; thin: 95% CI); dotted lines: ±0.01',
              fontsize=10)
save(fig, 'fig5_auroc_difference')

# Figures 6 and 7: ROC, precision-recall and calibration on the held-out test set.
tp = POOL['TEST']
cm = [m for m in ('R1', 'R2', 'R3', 'R4', 'R5', 'GRUD', 'TBAL') if m in tp]
if cm:
    fig, axes = plt.subplots(2, 1, figsize=(8.2 * CM, 15 * CM), layout='constrained')
    for m in cm:
        yv, sv = tp[m]['y'].values, tp[m]['score'].values
        fpr, tpr, _ = roc_curve(yv, sv)
        prec, rec, _ = precision_recall_curve(yv, sv)
        kw = dict(color=COLOR[m], ls=STYLE[m], lw=1.1)
        axes[0].plot(fpr, tpr, label=f"{LABEL.get(m, m)} ({roc_auc_score(yv, sv):.3f})", **kw)
        axes[1].plot(rec, prec, label=f"{LABEL.get(m, m)} ({pa.average_precision_score(yv, sv):.3f})", **kw)
    axes[0].plot([0, 1], [0, 1], color='0.6', lw=0.7, ls='--')
    axes[0].set(xlabel='False positive rate', ylabel='True positive rate', xlim=(0, 1), ylim=(0, 1.01))
    axes[1].axhline(tp[cm[0]]['y'].mean(), color='0.6', lw=0.7, ls='--')
    axes[1].set(xlabel='Recall', ylabel='Precision', xlim=(0, 1), ylim=(0, 1.01))
    # Opaque, compact legend boxes keep the diagonal and the prevalence line out of the text.
    box = dict(frameon=True, facecolor='white', edgecolor='none', framealpha=1, labelspacing=0.3)
    axes[0].legend(title='AUROC', loc='lower right', fontsize=8, title_fontsize=8, **box)
    axes[1].legend(title='AUPRC', loc='lower left', fontsize=8, title_fontsize=8, **box)
    for ax, t in zip(axes, ('(a)', '(b)')):
        ax.set_title(t, loc='left', fontsize=10)
    save(fig, 'fig6_roc_pr_test')

    ncol = 2
    nrow = int(np.ceil(len(cm) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(8.2 * CM, 3.6 * nrow * CM), sharex=True,
                             sharey=True, layout='constrained', squeeze=False)
    for ax, m in zip(axes.ravel(), cm):
        yv, sv = tp[m]['y'].values, tp[m]['score'].values
        obs, pred = calibration_curve(yv, sv, n_bins=10, strategy='quantile')
        cal = pa.calibration(yv, sv)
        ax.plot([0, 1], [0, 1], color='0.6', lw=0.7, ls='--')
        ax.plot(pred, obs, marker=MARK[m], ms=3, color=COLOR[m], lw=1.0)
        ax.set_title(f"{LABEL.get(m, m)} (slope {cal['calibration slope']:.2f})", fontsize=9)
        ax.set(xlim=(0, 1), ylim=(0, 1), xticks=[0, 0.5, 1], yticks=[0, 0.5, 1])
    for ax in axes.ravel()[len(cm):]:
        ax.axis('off')
    for ax in axes.ravel()[len(cm) - ncol:len(cm)]:    # lowest panel of each column
        ax.tick_params(labelbottom=True)
    fig.supxlabel('Predicted probability', fontsize=10)
    fig.supylabel('Observed proportion', fontsize=10)
    save(fig, 'fig7_calibration_test')


## 23. Export

A zip file with the aggregate tables, the figures, the run log, the selected hyperparameters, the code (`src/`), and the timing query.

Caches, per-patient predictions, and per-patient partition files are **not** included. Model weights are not saved either, since they derive from credentialed data; the code with its fixed seeds retrains them.

In [ ]:
# ============================================================
# 23. Export aggregate results and code (no patient-level files)
# ============================================================
import zipfile

out = f'{BASE}/paperA_revision_results.zip'
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as zf:
    for folder in (TAB_R, FIG_R):
        for f in sorted(glob.glob(f'{folder}/*')):
            if os.path.isfile(f):
                zf.write(f, os.path.relpath(f, BASE))
    for f in [LOG, f'{RES_R}/gate.json', f'{RES_R}/temporal_completed.json',
              *glob.glob(f'{RES_R}/selected_hp_*.json')]:
        if os.path.exists(f):
            zf.write(f, os.path.relpath(f, BASE))
    for f in ('paper_a_rev.py', 'train.py', 'windows.py', 'metrics.py'):
        if os.path.exists(f'{SRC}/{f}'):
            zf.write(f'{SRC}/{f}', f'code/src/{f}')
    zf.writestr('code/sql/sepsis3_timing.sql', pa.SEPSIS3_TIMING_SQL)
    names = zf.namelist()
assert not any(n.startswith(('cache_', 'results_paperA_rev/preds')) for n in names)
print(f'{len(names)} files, {os.path.getsize(out) / 1e6:.2f} MB -> {out}')
try:
    from google.colab import files
    files.download(out)
except ImportError:
    pass
